In [1]:
!pip install -q torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 --index-url https://download.pytorch.org/whl/cu128
!pip install -q transformers datasets seqeval evaluate tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.3 MB/s eta 0:00:00


In [ ]:
# =========================================
# 🧪 TEST USING TRAINED FUSION MODEL
# =========================================
import json
import torch
import numpy as np
from tqdm import tqdm
import evaluate

seqeval_metric = evaluate.load("seqeval")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

test_file_path = "/content/SE-cleaned_file.jsonl"

# =========================================
# 1️⃣ LOAD TRAINED MODEL
# =========================================
model = FusionNER().to(DEVICE)
model.load_state_dict(torch.load("fusion_iclr_model.pt", map_location=DEVICE))
model.eval()

# =========================================
# 2️⃣ LOAD DATA
# =========================================
dataset = []
with open(test_file_path, "r") as f:
    for line in f:
        dataset.append(json.loads(line))

texts = [d["input"].lower() for d in dataset]
true_entities = [json.loads(d["output"]) for d in dataset]

# =========================================
# 3️⃣ GROUND TRUTH → BIO
# =========================================
true_token_labels = []

for text, ents in zip(texts, true_entities):
    tokens, labels = align(text, ents)
    true_token_labels.append(labels)

# =========================================
# 4️⃣ PREDICTIONS (FUSION MODEL)
# =========================================
all_pred_labels = []
all_ents = []

for text in tqdm(texts):

    tokens = text.split()

    # encode like training
    enc = encode(tokens, ["O"] * len(tokens))

    input_ids = torch.tensor([enc["input_ids"]]).to(DEVICE)
    attention_mask = torch.tensor([enc["attention_mask"]]).to(DEVICE)
    input_ids_rob = torch.tensor([enc["input_ids_rob"]]).to(DEVICE)
    attention_mask_rob = torch.tensor([enc["attention_mask_rob"]]).to(DEVICE)

    with torch.no_grad():
        outputs = model(
            input_ids,
            attention_mask,
            input_ids_rob,
            attention_mask_rob,
            labels=None
        )

    # CRF decode → list of label ids
    pred_ids = outputs[0]

    pred_labels = []
    word_ids = sci_tok(
        tokens,
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    ).word_ids()

    prev_word = None

    for idx, word_id in enumerate(word_ids):

        if word_id is None:
            continue

        if word_id != prev_word:
            pred_labels.append(id2label[pred_ids[idx]])

        prev_word = word_id

    all_pred_labels.append(pred_labels)

    # =========================================
    # 🔁 BIO → ENTITY FORMAT
    # =========================================
    entities = []
    current = None

    for token, label in zip(tokens, pred_labels):

        if label.startswith("B-"):
            if current:
                entities.append(current)

            current = {
                "entity": token,
                "label": label[2:]
            }

        elif label.startswith("I-") and current:
            current["entity"] += " " + token

        else:
            if current:
                entities.append(current)
                current = None

    if current:
        entities.append(current)

    all_ents.append(entities)

# =========================================
# 5️⃣ ALIGN LENGTHS
# =========================================
aligned_preds = []
aligned_truths = []

for pred, true in zip(all_pred_labels, true_token_labels):
    min_len = min(len(pred), len(true))
    aligned_preds.append(pred[:min_len])
    aligned_truths.append(true[:min_len])

# =========================================
# 6️⃣ EVALUATION
# =========================================
results = seqeval_metric.compute(
    predictions=aligned_preds,
    references=aligned_truths
)

print("\n📊 FUSION MODEL RESULTS:")
print(f"Precision : {results['overall_precision']:.4f}")
print(f"Recall    : {results['overall_recall']:.4f}")
print(f"F1 Score  : {results['overall_f1']:.4f}")
print(f"Accuracy  : {results['overall_accuracy']:.4f}")

# =========================================
# 7️⃣ SAVE PREDICTIONS
# =========================================
output = []

for text, ents in zip(texts, all_ents):
    output.append({
        "text": text,
        "predicted_entities": ents
    })

with open("FUSION_PREDICTIONS.json", "w") as f:
    json.dump(output, f, indent=4)

print("\n✅ Saved: FUSION_PREDICTIONS.json")

100%|██████████| 30/30 [00:00<00:00, 41.30it/s]


📊 ENSEMBLE RESULTS ON NEW FILE:
Precision : 0.5200
Recall    : 0.0844
F1 Score  : 0.1453
Accuracy  : 0.8195

✅ Saved: NEW_FILE_PREDICTIONS.json



/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [7]:
# =========================================
# 🧪 RELATION EVAL + TRIPLET GENERATION PIPELINE
# =========================================

import json
import torch
import numpy as np
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# =========================================
# 1️⃣ DEVICE + MODEL
# =========================================
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

# =========================================
# 2️⃣ LOAD DATA
# =========================================
with open("/content/SE_textDocx_relations.json") as f:
    eval_data = json.load(f)

with open("/content/NEW_FILE_PREDICTIONS.json") as f:
    ner_data = json.load(f)

# =========================================
# 3️⃣ CLEAN ENTITIES
# =========================================
def clean_entities(entities):
    cleaned = []
    for e in entities:
        text = e["entity"].strip().lower()

        # remove junk
        if len(text) < 3:
            continue
        if text in ["the", "this", "that", "data"]:
            continue
        if text.isdigit():
            continue

        cleaned.append(e)

    return cleaned

# =========================================
# 4️⃣ RELATION PREDICTION FUNCTION
# =========================================
def predict_relation_with_conf(text, e1, t1, e2, t2):

    inp = f"Sentence: {text} Entity1: {e1} ({t1}). Entity2: {e2} ({t2})."

    inputs = tokenizer(
        inp,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        out = model(**inputs)
        probs = torch.softmax(out.logits, dim=1)
        conf, pred = torch.max(probs, dim=1)

    relation = id2label[pred.item()]
    confidence = conf.item()

    return relation, confidence

# =========================================
# 5️⃣ EVALUATION (SENTENCE LEVEL)
# =========================================
y_true = []
y_pred = []

for eval_item, ner_item in tqdm(zip(eval_data, ner_data), total=len(eval_data)):

    text = eval_item["input"]
    gold_rel = eval_item["output"].strip()

    entities = clean_entities(ner_item["predicted_entities"])

    predicted_rel = "no_relation"

    # pick FIRST meaningful relation
    for i in range(len(entities)):
        for j in range(i + 1, len(entities)):

            e1 = entities[i]["entity"]
            t1 = entities[i]["label"]
            e2 = entities[j]["entity"]
            t2 = entities[j]["label"]

            rel, conf = predict_relation_with_conf(text, e1, t1, e2, t2)

            if rel != "no_relation":
                predicted_rel = rel
                break

        if predicted_rel != "no_relation":
            break

    y_true.append(gold_rel)
    y_pred.append(predicted_rel)

# =========================================
# 6️⃣ METRICS
# =========================================
labels = sorted(list(set(y_true + y_pred)))

precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=labels, average="weighted", zero_division=0
)

acc = accuracy_score(y_true, y_pred)

print("\n📊 RELATION EVALUATION RESULTS")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

# =========================================
# 7️⃣ TRIPLET GENERATION (PAIR LEVEL)
# =========================================
triplets = []

for eval_item, ner_item in tqdm(zip(eval_data, ner_data), total=len(eval_data)):

    text = eval_item["input"]
    entities = clean_entities(ner_item["predicted_entities"])

    for i in range(len(entities)):
        for j in range(i + 1, len(entities)):

            e1 = entities[i]["entity"]
            t1 = entities[i]["label"]
            e2 = entities[j]["entity"]
            t2 = entities[j]["label"]

            # avoid same entity
            if e1 == e2:
                continue

            relation, confidence = predict_relation_with_conf(text, e1, t1, e2, t2)

            # 🔥 FILTERS (VERY IMPORTANT)
            if relation == "no_relation":
                continue

            if confidence < 0.6:   # tune this (0.5–0.7 best)
                continue

            triplets.append({
                "text": text,
                "entity1": e1,
                "entity1_type": t1,
                "relation": relation,
                "entity2": e2,
                "entity2_type": t2,
                "confidence": round(confidence, 3)
            })

# =========================================
# 8️⃣ SAVE OUTPUT
# =========================================
with open("/content/SE-FINAL_TRIPLETS.json", "w") as f:
    json.dump(triplets, f, indent=4)

print(f"\n✅ Saved {len(triplets)} triplets → SE-FINAL_TRIPLETS.json")

# =========================================
# 9️⃣ OPTIONAL: QUICK STATS
# =========================================
from collections import Counter

rel_counts = Counter([t["relation"] for t in triplets])

print("\n📊 Relation Distribution:")
for k, v in rel_counts.items():
    print(f"{k}: {v}")

100%|██████████| 26/26 [00:00<00:00, 301.21it/s]



📊 RELATION EVALUATION RESULTS
Accuracy : 0.2692
Precision: 0.1756
Recall   : 0.2692
F1 Score : 0.1532


100%|██████████| 26/26 [00:00<00:00, 287.77it/s]


✅ Saved 3 triplets → SE-FINAL_TRIPLETS.json

📊 Relation Distribution:
part_of: 2
implements: 1
